# Notebook status and evaluation note

This notebook is retained as a portfolio record of the original experiment. The workflow evaluates `test_loader` during training and uses test accuracy for checkpoint selection. Its stored metrics are therefore **historical development results**, not a final estimate from an untouched test set. A rigorous rerun should select checkpoints on a separate validation split and evaluate the test split once.

The original Google Colab and Google Drive paths are also retained and must be changed before running the notebook.


In [ ]:
# 0. Install required libraries
# =======================================================================
!pip install -q numpy pandas scikit-learn matplotlib seaborn torch torchvision qiskit qiskit-machine-learning qiskit-aer imbalanced-learn tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
# 1. Imports & Setup (QISKIT-SAFE)
# =======================================================================
import os, random, math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# -------- QISKIT (V2 ONLY - VERY IMPORTANT) --------
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit.quantum_info import SparsePauliOp


# -------- Metrics --------
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import shutil

In [ ]:
# 2. Reproducibility
# =======================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
# 3. Mount Google Drive & Load Dataset
# =======================================================================
from google.colab import drive
import zipfile

drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/chest_xray.zip"
extract_path = "/content/MyDrive/chest_xray"

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
print("Dataset ready at:", extract_path)

Mounted at /content/drive
Dataset ready at: /content/MyDrive/chest_xray


In [ ]:
base = "/content/MyDrive/chest_xray"
if os.path.isdir(os.path.join(base, "chest_xray")):
    base = os.path.join(base, "chest_xray")
print("Base:", base)
print("Contents:", os.listdir(base))
for split in ["train", "test", "val"]:
    sp = os.path.join(base, split)
    if os.path.exists(sp):
        for cls in os.listdir(sp):
            cp = os.path.join(sp, cls)
            if os.path.isdir(cp):
                print(f"  {split}/{cls}: {len(os.listdir(cp))} files")


Base: /content/MyDrive/chest_xray/chest_xray
Contents: ['val', 'train', 'test']
  train/PNEUMONIA: 3875 files
  train/NORMAL: 1341 files
  test/PNEUMONIA: 390 files
  test/NORMAL: 234 files
  val/PNEUMONIA: 8 files
  val/NORMAL: 8 files


In [ ]:
import shutil, zipfile

# Delete the incomplete extraction
extract_path = "/content/MyDrive/chest_xray"
if os.path.exists(extract_path):
    shutil.rmtree(extract_path)
    print("Deleted old extraction")

# Re-extract
zip_path = "/content/drive/MyDrive/chest_xray.zip"
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
    print("Extraction complete!")

# Verify
base = extract_path
if os.path.isdir(os.path.join(base, "chest_xray")):
    base = os.path.join(base, "chest_xray")
for split in ["train", "test", "val"]:
    sp = os.path.join(base, split)
    if os.path.exists(sp):
        for cls in sorted(os.listdir(sp)):
            cp = os.path.join(sp, cls)
            if os.path.isdir(cp):
                print(f"  {split}/{cls}: {len(os.listdir(cp))} files")
    else:
        print(f"  {split}/ — MISSING!")


Deleted old extraction
Extraction complete!
  train/NORMAL: 1341 files
  train/PNEUMONIA: 3875 files
  test/NORMAL: 234 files
  test/PNEUMONIA: 390 files
  val/NORMAL: 8 files
  val/PNEUMONIA: 8 files


In [ ]:
# 4. Merge train+test+val -> Balance -> 80-20 Split (ROBUST)
# =======================================================================
set_seed(42)

base = extract_path
if os.path.isdir(os.path.join(base, "chest_xray")):
    base = os.path.join(base, "chest_xray")

print("Using base dataset path:", base)

def find_class_dir(parent, class_name):
    for d in os.listdir(parent):
        if d.lower() == class_name.lower():
            return os.path.join(parent, d)
    raise FileNotFoundError(f"{class_name} not found in {parent}")

normal_dirs = []
pneumonia_dirs = []

for split in ["train", "test", "val"]:
    split_path = os.path.join(base, split)
    try:
        normal_dirs.append(find_class_dir(split_path, "normal"))
        pneumonia_dirs.append(find_class_dir(split_path, "pneumonia"))
    except FileNotFoundError:
        pass

# Collect image paths
normal_imgs = []
for d in normal_dirs:
    normal_imgs += [os.path.join(d, f) for f in os.listdir(d)]

pneumonia_imgs = []
for d in pneumonia_dirs:
    pneumonia_imgs += [os.path.join(d, f) for f in os.listdir(d)]

print("Original counts -> Normal:", len(normal_imgs), "Pneumonia:", len(pneumonia_imgs))

# -------- BALANCING --------
pneumonia_imgs = random.sample(pneumonia_imgs, len(normal_imgs))

# -------- SPLIT --------
norm_train, norm_test = train_test_split(normal_imgs, test_size=0.2, random_state=42)
pneu_train, pneu_test = train_test_split(pneumonia_imgs, test_size=0.2, random_state=42)

# Output folders
balanced_base = "/content/balanced_chest_xray"
train_dir = f"{balanced_base}/train"
test_dir = f"{balanced_base}/test"

for cls in ["NORMAL", "PNEUMONIA"]:
    os.makedirs(f"{train_dir}/{cls}", exist_ok=True)
    os.makedirs(f"{test_dir}/{cls}", exist_ok=True)

def copy_files(files, dst):
    for f in files:
        shutil.copy(f, dst)

copy_files(norm_train, f"{train_dir}/NORMAL")
copy_files(norm_test, f"{test_dir}/NORMAL")
copy_files(pneu_train, f"{train_dir}/PNEUMONIA")
copy_files(pneu_test, f"{test_dir}/PNEUMONIA")

print("Balanced dataset created")
print("Train -> Normal:", len(norm_train), "Pneumonia:", len(pneu_train))
print("Test  -> Normal:", len(norm_test), "Pneumonia:", len(pneu_test))


Using base dataset path: /content/MyDrive/chest_xray/chest_xray
Original counts -> Normal: 1583 Pneumonia: 4273
Balanced dataset created
Train -> Normal: 1266 Pneumonia: 1266
Test  -> Normal: 317 Pneumonia: 317


In [ ]:
import random # Ensure random is imported for random.sample

# 5. Load Images into Memory (Balanced Dataset)
# =======================================================================
img_size = (224, 224)

def load_images_from_folder(folder, label, target_size=img_size):
    data_labels = [] # Store (image_data, label) tuples
    skipped_count = 0
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        try:
            img = Image.open(path).convert("RGB").resize(target_size)
            data_labels.append((np.array(img), label))
        except Exception as e:
            skipped_count += 1
    if skipped_count > 0:
        print(f"WARNING: Skipped {skipped_count} images in {folder} due to loading errors.")
    return data_labels

print("Loading Test & Train Images in memory...")

# Load images and labels for training set
train_norm_data_labels = load_images_from_folder(f"{train_dir}/NORMAL", 0)
train_pneu_data_labels = load_images_from_folder(f"{train_dir}/PNEUMONIA", 1)

# Balance the training set after loading if any images were skipped
min_train_count = min(len(train_norm_data_labels), len(train_pneu_data_labels))
if len(train_norm_data_labels) > min_train_count:
    train_norm_data_labels = random.sample(train_norm_data_labels, min_train_count)
if len(train_pneu_data_labels) > min_train_count:
    train_pneu_data_labels = random.sample(train_pneu_data_labels, min_train_count)

X_train = np.array([img for img, _ in (train_norm_data_labels + train_pneu_data_labels)])
y_train = np.array([label for _, label in (train_norm_data_labels + train_pneu_data_labels)])

# Load images and labels for test set
test_norm_data_labels = load_images_from_folder(f"{test_dir}/NORMAL", 0)
test_pneu_data_labels = load_images_from_folder(f"{test_dir}/PNEUMONIA", 1)

# Balance the test set after loading if any images were skipped
min_test_count = min(len(test_norm_data_labels), len(test_pneu_data_labels))
if len(test_norm_data_labels) > min_test_count:
    test_norm_data_labels = random.sample(test_norm_data_labels, min_test_count)
if len(test_pneu_data_labels) > min_test_count:
    test_pneu_data_labels = random.sample(test_pneu_data_labels, min_test_count)

X_test = np.array([img for img, _ in (test_norm_data_labels + test_pneu_data_labels)])
y_test = np.array([label for _, label in (test_norm_data_labels + test_pneu_data_labels)])


print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

assert Counter(y_train)[0] == Counter(y_train)[1], "Train set NOT balanced!"
assert Counter(y_test)[0] == Counter(y_test)[1], "Test set NOT balanced!"
print("Sanity check passed: datasets are balanced")


Loading Test & Train Images in memory...
Train shape: (2532, 224, 224, 3) Test shape: (634, 224, 224, 3)
Sanity check passed: datasets are balanced


In [ ]:
# 5.1 Dataset Class & DataLoaders
# =======================================================================
class ImageDataset(Dataset):
    def __init__(self, X, y, train=True):
        self.X = X
        self.y = y
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)) if train else transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip() if train else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx]
        label = self.y[idx]
        img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(ImageDataset(X_train, y_train, train=True), batch_size=16, shuffle=True)
test_loader = DataLoader(ImageDataset(X_test, y_test, train=False), batch_size=16, shuffle=False)


In [ ]:
# 6. Define Models (Baseline + NEW QUANTUM)
# =======================================================================
def create_qnn(num_qubits=4, reps=2):
    feature_map = zz_feature_map(num_qubits, reps=reps)
    ansatz = real_amplitudes(num_qubits, reps=reps)
    qc = feature_map.compose(ansatz)

    estimator = StatevectorEstimator()

    # -------------------------------------------------------------
    # NEW: Create 4 Observables (one for each qubit) instead of 1
    # This generates ["ZIII", "IZII", "IIZI", "IIIZ"]
    # -------------------------------------------------------------
    observables = []
    for i in range(num_qubits):
        obs_str = "I"*i + "Z" + "I"*(num_qubits - 1 - i)
        observables.append(SparsePauliOp(obs_str))

    qnn = EstimatorQNN(
        circuit=qc,
        estimator=estimator,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        observables=observables
    )
    return TorchConnector(qnn)


class QuantumResNet(nn.Module):
    def __init__(self, num_classes=2, num_qubits=4, q_reps=2, proj_dim=8):
        super().__init__()

        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.resnet.fc = nn.Identity()  # Outputs 512 dimensions

        # ---------------------------------------------------------
        # NEW: Classical Bottleneck
        # Compress the 512 dimensions down to 32
        # ---------------------------------------------------------
        self.bottleneck = nn.Linear(512, 32)

        # ---------------------------------------------------------
        # Quantum Branch
        # ---------------------------------------------------------
        self.proj = nn.Linear(512, proj_dim) # Project onto quantum states
        self.proj_norm = nn.LayerNorm(proj_dim)

        self.qnn = create_qnn(num_qubits=num_qubits, reps=q_reps)
        self.qnorm = nn.LayerNorm(4)  # Now norming 4 outputs

        # ---------------------------------------------------------
        # The Final Fusion: 32 classical + 4 quantum = 36 features
        # ---------------------------------------------------------
        self.fc = nn.Linear(32 + 4, num_classes)

    def forward(self, x):
        x_feat = self.resnet(x) # [Batch, 512]

        # Branch 1: Classical feature compression
        c_out = torch.relu(self.bottleneck(x_feat))  # [Batch, 32]

        # Branch 2: Quantum path
        q_in = torch.tanh(self.proj_norm(self.proj(x_feat)))
        q_out = self.qnorm(self.qnn(q_in)) # [Batch, 4]

        # Final Fusion
        combined = torch.cat([c_out, q_out], dim=1) # [Batch, 36]
        return self.fc(combined)

In [ ]:
# 7 - 10. Training Utilities & Evaluation
# =======================================================================
def compute_test_loss(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(dataloader)

def make_optimizer(model):
    q_params = [p for n, p in model.named_parameters() if 'qnn' in n or 'proj' in n]
    base_params = [p for n, p in model.named_parameters() if not any(k in n for k in ['qnn', 'proj'])]
    optimizer = torch.optim.AdamW([
        {'params': base_params, 'lr': 3e-4},
        {'params': q_params, 'lr': 1e-3},
    ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    return optimizer, scheduler

def evaluate_acc(model, dataloader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

def train_model(model, train_loader, test_loader, device, epochs=50, warmup_epochs=5, save_path=None):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer, scheduler = make_optimizer(model)
    best_acc = 0; hist = {"loss": [], "acc": [], "test_acc": [], "test_loss": []}

    if hasattr(model, "resnet"):
        print(f"\nWarm-up for {warmup_epochs} epochs...")
        for p in model.resnet.parameters(): p.requires_grad = False
        for ep in range(warmup_epochs):
            model.train(); run_loss = 0; correct = 0; total = 0
            for x, y in tqdm(train_loader, desc=f"Warmup {ep+1}/{warmup_epochs}"):
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(x); loss = criterion(out, y); loss.backward(); optimizer.step()
                run_loss += loss.item(); _, pred = out.max(1)
                correct += pred.eq(y).sum().item(); total += y.size(0)
            print(f"Warmup Epoch {ep+1}: Loss={{run_loss/len(train_loader):.4f}} Acc={{100*correct/total:.2f}}%")
        for p in model.resnet.parameters(): p.requires_grad = True

    print("\nFull Training...")
    for ep in range(epochs):
        model.train(); run_loss = 0; correct = 0; total = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x); loss = criterion(out, y); loss.backward(); optimizer.step()
            run_loss += loss.item(); _, pred = out.max(1)
            correct += pred.eq(y).sum().item(); total += y.size(0)
        scheduler.step()

        train_acc = 100 * correct / total
        test_acc = evaluate_acc(model, test_loader, device)
        test_loss = compute_test_loss(model, test_loader, criterion, device)

        hist["loss"].append(run_loss/len(train_loader)); hist["acc"].append(train_acc);
        hist["test_acc"].append(test_acc); hist["test_loss"].append(test_loss)

        if test_acc > best_acc and save_path:
            best_acc = test_acc; torch.save(model.state_dict(), save_path)
            print(f"Saved best model ({best_acc:.2f}%)")
        print(f"Epoch {ep+1}: Loss={{run_loss/len(train_loader):.4f}}, TrainAcc={{train_acc:.2f}}%, TestAcc={{test_acc:.2f}}%")

    return hist, best_acc

def plot_training(hist):
    ep = range(1, len(hist["loss"]) + 1)
    plt.figure(figsize=(18, 5))

    plt.subplot(1, 3, 1)
    plt.plot(ep, hist["loss"], 'r-o', label="Train Loss")
    plt.plot(ep, hist["test_loss"], 'b-o', label="Test Loss")
    plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(ep, hist["acc"], 'g-o', label="Train Acc")
    plt.plot(ep, hist["test_acc"], 'm-o', label="Test Acc")
    plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# 11. Train Quantum Hybrid ResNet (32+4 ARCHITECTURE)
# =======================================================================
quantum_path = "/content/drive/MyDrive/best_quantum_32plus4_3w15e.pth"
q_model = QuantumResNet(num_classes=2, num_qubits=4, q_reps=2, proj_dim=4).to(device)
q_hist, q_acc = train_model(q_model, train_loader, test_loader, device, epochs=15, warmup_epochs=3, save_path=quantum_path)
plot_training(q_hist)
print("Quantum Hybrid Final Test Acc:", q_acc)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 207MB/s]



Warm-up for 3 epochs...


Warmup 1/3: 100%|██████████| 159/159 [17:12<00:00,  6.50s/it]


Warmup Epoch 1: Loss={run_loss/len(train_loader):.4f} Acc={100*correct/total:.2f}%


...

Epoch 13/15: 100%|██████████| 159/159 [17:14<00:00,  6.51s/it]


Epoch 13: Loss={run_loss/len(train_loader):.4f}, TrainAcc={train_acc:.2f}%, TestAcc={test_acc:.2f}%


Epoch 14/15: 100%|██████████| 159/159 [17:14<00:00,  6.50s/it]


Epoch 14: Loss={run_loss/len(train_loader):.4f}, TrainAcc={train_acc:.2f}%, TestAcc={test_acc:.2f}%


Epoch 15/15:  14%|█▍        | 23/159 [02:27<14:09,  6.24s/it]

In [ ]:
print("Evaluating the loaded model to find its test accuracy...")

# Evaluate the loaded model
# The evaluate_acc function was defined in a previous cell (8juBMSGzYP3y)
best_accuracy_from_saved_model = evaluate_acc(loaded_q_model, test_loader, device)

print(f"Best Test Accuracy from the saved model: {best_accuracy_from_saved_model:.2f}%")

Evaluating the loaded model to find its test accuracy...


NameError: name 'evaluate_acc' is not defined

In [ ]:
# =======================================================================
# 14. Sensitivity, Specificity, Precision, F1, Classification Report
# =======================================================================
q_model.eval()
y_true, y_pred, y_score = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = q_model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        y_score.extend(probs)
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_score = np.array(y_score)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

sensitivity = 100.0 * tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = 100.0 * tn / (tn + fp) if (tn + fp) > 0 else 0
precision = 100.0 * tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
accuracy = 100.0 * (tp + tn) / (tp + tn + fp + fn)

print(f"\n{'='*60}")
print(f"  CLASSIFICATION METRICS — 32+4 Quantum Hybrid (3W+15E)")
print(f"{'='*60}")
print(f"  Accuracy:    {accuracy:.4f}%")
print(f"  Sensitivity: {sensitivity:.4f}% (Recall for PNEUMONIA)")
print(f"  Specificity: {specificity:.4f}% (Recall for NORMAL)")
print(f"  Precision:   {precision:.4f}%")
print(f"  F1-Score:    {f1:.4f}%")
print(f"  ROC AUC:     {roc_auc:.4f}")
print(f"{'='*60}")
print(f"  TP: {tp}  |  FP: {fp}")
print(f"  FN: {fn}  |  TN: {tn}")
print(f"{'='*60}")

print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"]))

NameError: name 'q_model' is not defined